# MNIST: model comparison & Explainable AI

Workflow mới dùng package `mnist_project`, cùng train/validation split cho mọi
model và giữ riêng official test. Mặc định chạy nhỏ (CNN, 1 epoch) để thử luồng.
Xem README cho lệnh full-data và Optuna. Kết quả của notebook này không phải
các số lịch sử đã lưu trong `reports/notebook/`.

Cài từ thư mục project: `python -m pip install -r requirements.txt`.
Chọn kernel từ môi trường đã cài project. Chạy các cell theo thứ tự.


In [ ]:
from pathlib import Path
import os
import json
from datetime import datetime, timezone
import numpy as np
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Open this notebook from the project root or notebooks directory.")
os.chdir(ROOT)

from mnist_project.data import load_data
from mnist_project.eda import create_eda
from mnist_project.experiments import run_experiment, predict_images
from mnist_project.explain import explain_run

QUICK = True
# To run offline, set MNIST_DATA to the path of a Keras-style MNIST NPZ.
DATA_PATH = os.environ.get("MNIST_DATA") or None
RUN = ROOT / "artifacts" / ("notebook-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
print("Output:", RUN)


## 1. Dữ liệu

Mặc định: chia 60.000 training ảnh thành 54.000 train / 6.000 validation;
10.000 official test giữ riêng. QUICK lấy một tập con có phân tầng của train
và test, validation vẫn đầy đủ. Lần đầu cần Internet để tải MNIST nếu chưa có.


In [ ]:
data = load_data(DATA_PATH, seed=42, train_limit=2000 if QUICK else None,
                 test_limit=500 if QUICK else None)
print({"train": len(data.y_train), "validation": len(data.y_val), "test": len(data.y_test)})
assert not set(data.train_indices) & set(data.val_indices)
assert data.x_train.dtype == np.float32


## 2. EDA

Lưu phân bố lớp, ví dụ từng chữ số, ảnh trung bình, histogram pixel và độ thưa.


In [ ]:
summary = create_eda(data, RUN / "eda")
display(Image(filename=str(RUN / "eda" / "samples_and_means.png")))


## 3. Train và đánh giá

Chạy một CNN để minh họa. Đổi MODEL thành `knn`, `svm`, `random_forest`, `mlp`,
`resnet9`, `tinyvit` để chạy mô hình khác. Với benchmark cả bảy, dùng CLI
`python -m mnist_project train --model all --output artifacts/full-run`.
QUICK và dữ liệu nhân tạo chỉ kiểm tra luồng, không đánh giá chất lượng model.


In [ ]:
MODEL = "cnn"
model_dir = RUN / MODEL
metrics = run_experiment(MODEL, data, model_dir, epochs=1 if QUICK else 15, batch_size=64)
{k: metrics[k] for k in ["accuracy", "f1_macro", "f1_weighted", "n_test", "train_seconds"]}


In [ ]:
display(Image(filename=str(model_dir / "confusion_matrix.png")))
if (model_dir / "learning_curves.png").exists():
    display(Image(filename=str(model_dir / "learning_curves.png")))

## 4. Lưu/nạp và dự đoán

Runner đã lưu model và metadata. predict_images tự nạp lại từ đĩa. SVM trả
`decision_scores` thay vì xác suất. Không nạp model từ nguồn không tin cậy.


In [ ]:
result = predict_images(model_dir, data.x_test[:5])
print("True:", data.y_test[:5].tolist())
print("Pred:", result["predictions"])


## 5. Giải thích

`auto` chọn phương pháp thích hợp cho MODEL. Với CNN/ResNet dùng Grad-CAM,
Tiny-ViT dùng attention rollout, MLP dùng IG. Có thể dùng `method="ig"` cho
bất kỳ deep model nào để xem attribution có dấu và deletion test trên một ảnh.
Ảnh chưa train đủ có thể cho map yếu hoặc rỗng; map không chứng minh đúng/sai.


In [ ]:
explanation = explain_run(model_dir, data.x_test[:1], RUN / "explanation", method="auto")
display(Image(filename=str(RUN / "explanation" / "explanation.png")))
explanation


## 6. Optuna (tùy chọn, không tự chạy)

Chỉ tối ưu trên train và validation. Giữ cùng seed/split/subset khi train từ
best_params. Không dùng official test để lựa chọn hyperparameter.

```python
from mnist_project.tuning import tune
best = tune("cnn", data, RUN / "search", trials=5, epochs=10)
print(best["best_params"])
```

Kết quả fresh run lưu trong artifacts/, còn kết quả lịch sử ở reports/notebook/.
Đọc docs/methodology.md trước khi so sánh hai nguồn kết quả này.
